# 第305章: GAN潜在空間の構造

## 📋 この章で学ぶこと

この章を終えると、以下ができるようになります：

- [ ] GANの潜在空間がVAEと構造的にどう異なるかを説明できる
- [ ] Z空間（入力ノイズ空間）とW空間（StyleGANの中間空間）の違いを理解できる
- [ ] モード崩壊が潜在空間に与える影響を説明できる
- [ ] シンプルなGANを実装し、その潜在空間をVAEと比較できる

## 🎯 前提知識

- ✅ Notebook 300-304（潜在空間の基礎、Phase 1全体）
- ✅ Notebook 37-38（VAE理論と実装）
- ✅ GANの基本概念（Generator / Discriminator）

⏱️ **推定学習時間**: 90-120分
📊 **難易度**: ★★★☆☆（中級）
🎓 **カテゴリ**: 理論・実践

---

## 🌟 はじめに — Phase 2: GANベースの画像変容へ

Phase 1ではVAEの潜在空間を深く探索しました。
Phase 2では、**GAN（Generative Adversarial Network）** の潜在空間に焦点を当てます。

### VAE vs GAN の潜在空間

```
  VAE                                GAN
  ───                                ───
  入力画像 → エンコーダ → z → デコーダ → 再構成    z（ランダムノイズ）→ Generator → 生成画像
                                                                        ↕
                                                             Discriminator（本物/偽物判定）

  ✅ エンコーダがある → 画像→z変換が可能    ❌ エンコーダがない → 画像→z変換は別途必要
  △ 画像がぼやけがち                       ✅ シャープな高品質画像
  ✅ 潜在空間が正則化されている              △ 潜在空間の構造は学習任せ
```

### 📝 この章の構成

1. **GANの基本構造** — Generator と Discriminator のおさらい
2. **GANの学習と潜在空間** — MNISTでGANを学習
3. **GAN vs VAEの潜在空間比較** — 構造の違いを可視化
4. **Z空間とW空間** — StyleGANの革新的なアイデア
5. **モード崩壊と潜在空間** — GANの弱点と影響

In [ ]:
# ============================================================
# 環境設定
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import warnings

warnings.filterwarnings('ignore')

def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'MS Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)

print(f"日本語フォント: {font_used}")
print(f"Device: {device}")
print("✅ 環境設定完了")

---

## 1. GANの基本構造

### 🤔 GANの核心アイデア

GANは2つのネットワークが**敵対的に**学習します：

```
  ┌──────────────┐          ┌──────────────────┐
  │  Generator G │          │ Discriminator D   │
  │              │  偽画像   │                  │
  │  z (ノイズ)  │ ───────→ │ 本物？偽物？      │
  │  → 画像生成  │          │                  │
  └──────────────┘          │ 本物画像         │
                    ───────→ │ も判定           │
                             └──────────────────┘

  G の目標: Dを騙す偽画像を作る
  D の目標: 本物と偽物を見分ける
```

### GAN vs VAE — 潜在空間の根本的な違い

| 特性 | VAE | GAN |
|------|-----|-----|
| 潜在空間の制約 | KL正則化でN(0,1)に近づける | 制約なし（入力がN(0,1)なだけ） |
| エンコーダ | ある | ない（標準GAN） |
| 画像品質 | ぼやけがち | シャープ |
| 潜在空間の構造 | 滑らか・連続的 | 構造的だが保証なし |
| 学習の安定性 | 安定 | 不安定（モード崩壊etc） |

In [ ]:
# ============================================================
# シンプルなGANの実装
# ============================================================

class Generator(nn.Module):
    """潜在ベクトル z から画像を生成"""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            nn.Linear(512, 784),
            nn.Tanh(),  # [-1, 1]
        )

    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    """画像が本物か偽物かを判定"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

latent_dim_gan = 64
G = Generator(latent_dim_gan).to(device)
D = Discriminator().to(device)

print(f"✅ GAN構築完了")
print(f"   Generator: {sum(p.numel() for p in G.parameters()):,} パラメータ")
print(f"   Discriminator: {sum(p.numel() for p in D.parameters()):,} パラメータ")
print(f"   潜在次元: {latent_dim_gan}")

In [ ]:
# ============================================================
# GANの学習
# ============================================================

# データ（GANでは[-1,1]に正規化）
transform_gan = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # [0,1] → [-1,1]
])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform_gan)
train_loader_gan = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

# オプティマイザ
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = nn.BCELoss()

# 学習ループ
n_epochs = 30
g_losses, d_losses = [], []

print("GANの学習を開始...")
print("=" * 50)

for epoch in range(n_epochs):
    g_loss_epoch, d_loss_epoch = 0, 0

    for real_imgs, _ in train_loader_gan:
        batch_size = real_imgs.size(0)
        real_imgs = real_imgs.view(batch_size, -1).to(device)

        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # --- Discriminator学習 ---
        z = torch.randn(batch_size, latent_dim_gan).to(device)
        fake_imgs = G(z)
        d_real = D(real_imgs)
        d_fake = D(fake_imgs.detach())
        d_loss = criterion(d_real, real_labels) + criterion(d_fake, fake_labels)

        opt_D.zero_grad()
        d_loss.backward()
        opt_D.step()

        # --- Generator学習 ---
        z = torch.randn(batch_size, latent_dim_gan).to(device)
        fake_imgs = G(z)
        g_loss = criterion(D(fake_imgs), real_labels)

        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()

        g_loss_epoch += g_loss.item()
        d_loss_epoch += d_loss.item()

    n_batches = len(train_loader_gan)
    g_losses.append(g_loss_epoch / n_batches)
    d_losses.append(d_loss_epoch / n_batches)

    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/{n_epochs} | G Loss: {g_losses[-1]:.4f} | D Loss: {d_losses[-1]:.4f}")

print("=" * 50)
print("✅ GAN学習完了")

In [ ]:
# ============================================================
# GAN生成画像の確認
# ============================================================

G.eval()

fig, axes = plt.subplots(4, 8, figsize=(14, 7))

with torch.no_grad():
    z = torch.randn(32, latent_dim_gan).to(device)
    fake_imgs = G(z).cpu().numpy()

for i in range(32):
    row, col = i // 8, i % 8
    img = fake_imgs[i].reshape(28, 28)
    img = (img + 1) / 2  # [-1,1] → [0,1]
    axes[row, col].imshow(img, cmap='gray')
    axes[row, col].axis('off')

fig.suptitle('GANが生成した画像（ランダムなzから）', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_305_01_gan_generated.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 VAEよりシャープだが、品質にバラつきがある")
print("   → GANの潜在空間はVAEほど均一ではない")

---

## 2. GAN vs VAE の潜在空間比較

GANにはエンコーダがないため、画像→zの対応が直接求められません。
しかし、「Generatorの出力がzによってどう変わるか」を分析できます。

### 潜在空間の「滑らかさ」テスト

2つの z 点を選び、その間を補間してみます：
- **VAE**: 基本的に滑らかな補間ができる（KL正則化のおかげ）
- **GAN**: 滑らかかもしれないが保証はない

In [ ]:
# ============================================================
# VAEも学習して比較用に用意
# ============================================================

class VAE(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(logvar)
        return self.decode(z), mu, logvar

# VAE学習
transform_vae = transforms.ToTensor()
train_ds_vae = datasets.MNIST(root='./data', train=True, download=True, transform=transform_vae)
train_loader_vae = DataLoader(train_ds_vae, batch_size=256, shuffle=True)

vae = VAE(latent_dim=64).to(device)
opt_vae = optim.Adam(vae.parameters(), lr=1e-3)

print("比較用VAE (z=64) 学習中...")
for epoch in range(15):
    vae.train()
    for x, _ in train_loader_vae:
        x = x.view(-1, 784).to(device)
        recon, mu, logvar = vae(x)
        loss = nn.functional.binary_cross_entropy(recon, x, reduction='sum') \
               - 0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        opt_vae.zero_grad()
        loss.backward()
        opt_vae.step()
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1}/15")

print("✅ VAE学習完了")

In [ ]:
# ============================================================
# 補間の比較: GAN vs VAE
# ============================================================

G.eval()
vae.eval()

n_interp = 10

fig, axes = plt.subplots(4, n_interp, figsize=(16, 7))
alphas = np.linspace(0, 1, n_interp)

# GAN: 2組の補間
for row in range(2):
    z1 = torch.randn(1, latent_dim_gan).to(device)
    z2 = torch.randn(1, latent_dim_gan).to(device)

    with torch.no_grad():
        for i, alpha in enumerate(alphas):
            z_interp = (1 - alpha) * z1 + alpha * z2
            img = G(z_interp).cpu().numpy().reshape(28, 28)
            img = (img + 1) / 2
            axes[row, i].imshow(img, cmap='gray')
            axes[row, i].axis('off')
            if i == 0:
                label = 'GAN' if row == 0 else 'GAN'
                axes[row, i].set_ylabel(label, fontsize=12, rotation=0, labelpad=30)
            if row == 0:
                axes[row, i].set_title(f'{alpha:.1f}', fontsize=9)

# VAE: 2組の補間（テスト画像から）
test_ds_vae = datasets.MNIST(root='./data', train=False, transform=transform_vae)
test_loader_vae = DataLoader(test_ds_vae, batch_size=256, shuffle=False)

vae.eval()
test_batch = next(iter(test_loader_vae))[0]

for row_offset in range(2):
    row = row_offset + 2
    x1 = test_batch[row_offset * 10].view(1, 784).to(device)
    x2 = test_batch[row_offset * 10 + 5].view(1, 784).to(device)

    with torch.no_grad():
        mu1, _ = vae.encode(x1)
        mu2, _ = vae.encode(x2)

        for i, alpha in enumerate(alphas):
            z_interp = (1 - alpha) * mu1 + alpha * mu2
            img = vae.decode(z_interp).cpu().numpy().reshape(28, 28)
            axes[row, i].imshow(img, cmap='gray')
            axes[row, i].axis('off')
            if i == 0:
                axes[row, i].set_ylabel('VAE', fontsize=12, rotation=0, labelpad=30)

fig.suptitle('線形補間の比較: GAN（上2行）vs VAE（下2行）',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_305_02_interpolation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 GAN: ランダムなz間の補間 — 途中で品質が落ちることがある")
print("   VAE: エンコードしたz間の補間 — 比較的滑らか（KL正則化のおかげ）")

---

## 3. Z空間とW空間 — StyleGANの革新

### 🤔 なぜZ空間は構造が悪いのか？

標準GANのZ空間（入力ノイズ）は**球面上の正規分布**です。
しかし、学習データの分布は複雑で非球面的なため、
Z空間の構造とデータの意味構造に**ミスマッチ**が生じます。

### StyleGANの解決策: W空間

```
  標準GAN:   z ∈ N(0, I)  ──→  Generator  ──→  画像
                                    ↑
  StyleGAN:  z ∈ N(0, I)  ──→  Mapping Network (8層MLP)  ──→  w ∈ W  ──→  Synthesis Network  ──→  画像
                                                                  ↑
                                                        「W空間」= より構造化された潜在空間
```

| 空間 | 分布の形 | 構造 | 操作性 |
|------|---------|------|--------|
| Z空間 | 固定（正規分布） | 球面上に制約 | 限定的 |
| W空間 | 学習される | データに適応 | 高い |

### W空間の利点

1. **Disentanglement**: W空間のほうが属性が分離されやすい
2. **補間品質**: W空間での補間のほうが滑らか
3. **編集性**: 属性ベクトルがより正確に機能する

In [ ]:
# ============================================================
# Mapping Network（StyleGANのW空間）のシミュレーション
# 8層MLPでZ空間→W空間への変換を模倣
# ============================================================

class MappingNetwork(nn.Module):
    """Z空間からW空間への写像（StyleGAN風）"""
    def __init__(self, latent_dim=64, n_layers=8):
        super().__init__()
        layers = []
        for _ in range(n_layers):
            layers.extend([nn.Linear(latent_dim, latent_dim), nn.LeakyReLU(0.2)])
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

class StyleGenerator(nn.Module):
    """簡易StyleGAN: MappingNetwork + SynthesisNetwork"""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.mapping = MappingNetwork(latent_dim)
        self.synthesis = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            nn.Linear(512, 784),
            nn.Tanh(),
        )

    def forward(self, z, return_w=False):
        w = self.mapping(z)
        img = self.synthesis(w)
        if return_w:
            return img, w
        return img

styleG = StyleGenerator(latent_dim_gan).to(device)

# 学習
opt_styleG = optim.Adam(styleG.parameters(), lr=2e-4, betas=(0.5, 0.999))
D_style = Discriminator().to(device)
opt_D_style = optim.Adam(D_style.parameters(), lr=2e-4, betas=(0.5, 0.999))

print("簡易StyleGANの学習中...")
for epoch in range(30):
    for real_imgs, _ in train_loader_gan:
        bs = real_imgs.size(0)
        real_imgs = real_imgs.view(bs, -1).to(device)
        real_lab = torch.ones(bs, 1).to(device)
        fake_lab = torch.zeros(bs, 1).to(device)

        z = torch.randn(bs, latent_dim_gan).to(device)
        fake_imgs = styleG(z)
        d_loss = criterion(D_style(real_imgs), real_lab) + criterion(D_style(fake_imgs.detach()), fake_lab)
        opt_D_style.zero_grad(); d_loss.backward(); opt_D_style.step()

        z = torch.randn(bs, latent_dim_gan).to(device)
        fake_imgs = styleG(z)
        g_loss = criterion(D_style(fake_imgs), real_lab)
        opt_styleG.zero_grad(); g_loss.backward(); opt_styleG.step()

    if (epoch+1) % 10 == 0:
        print(f"  Epoch {epoch+1}/30")

print("✅ 簡易StyleGAN学習完了")

In [ ]:
# ============================================================
# Z空間 vs W空間の分布を可視化
# ============================================================

from sklearn.decomposition import PCA

styleG.eval()
n_samples = 3000

with torch.no_grad():
    z_samples = torch.randn(n_samples, latent_dim_gan).to(device)
    _, w_samples = styleG(z_samples, return_w=True)
    z_np = z_samples.cpu().numpy()
    w_np = w_samples.cpu().numpy()

# PCAで2次元に射影
pca_z = PCA(n_components=2)
pca_w = PCA(n_components=2)
z_2d = pca_z.fit_transform(z_np)
w_2d = pca_w.fit_transform(w_np)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(z_2d[:, 0], z_2d[:, 1], alpha=0.3, s=5, c='steelblue')
axes[0].set_title('Z空間（入力ノイズ）\n等方的な正規分布', fontsize=14, fontweight='bold')
axes[0].set_xlabel('PC1', fontsize=12)
axes[0].set_ylabel('PC2', fontsize=12)
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(w_2d[:, 0], w_2d[:, 1], alpha=0.3, s=5, c='coral')
axes[1].set_title('W空間（Mapping Network後）\n非等方的・データに適応した分布', fontsize=14, fontweight='bold')
axes[1].set_xlabel('PC1', fontsize=12)
axes[1].set_ylabel('PC2', fontsize=12)
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Z空間 vs W空間の分布比較（PCA射影）', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_305_03_z_vs_w_space.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 Z空間: 等方的な球状分布（入力の制約通り）")
print("   W空間: 歪んだ分布 → データの構造を反映している")
print("   W空間のほうが意味的な操作に適している理由がここにある")

---

## 4. モード崩壊と潜在空間への影響

### 🤔 モード崩壊（Mode Collapse）とは？

GANの学習でGeneratorが**データの一部のモード（種類）しか生成しなくなる**現象です。

```
  正常な学習:                    モード崩壊:
  z → 0, 1, 2, 3, ..., 9       z → 1, 1, 7, 1, 7, 1, ...
  （すべての数字を生成）          （一部の数字しか生成しない）
```

これが起きると、潜在空間の構造も歪みます：
- 生成しない数字に対応するz領域が「空白」になる
- 補間が不自然になる

In [ ]:
# ============================================================
# モード崩壊の可視化
# 生成画像の多様性を分析
# ============================================================

G.eval()

# 大量に生成して多様性を確認
n_gen = 200
with torch.no_grad():
    z = torch.randn(n_gen, latent_dim_gan).to(device)
    generated = G(z).cpu().numpy()

# 生成画像のピクセル値の分散で多様性を測定
gen_imgs = generated.reshape(n_gen, 28, 28)
gen_imgs = (gen_imgs + 1) / 2

# 画像間のペアワイズ距離
from itertools import combinations
sample_indices = np.random.choice(n_gen, 50, replace=False)
pairwise_dists = []
for i, j in combinations(sample_indices, 2):
    dist = np.mean((gen_imgs[i] - gen_imgs[j]) ** 2)
    pairwise_dists.append(dist)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左: 生成画像のサンプル
for idx in range(40):
    row, col = idx // 10, idx % 10
    ax = fig.add_subplot(2, 4, 1)  # dummy

# 生成画像のグリッド表示
grid_fig, grid_axes = plt.subplots(5, 10, figsize=(14, 7))
for idx in range(50):
    row, col = idx // 10, idx % 10
    grid_axes[row, col].imshow(gen_imgs[idx], cmap='gray')
    grid_axes[row, col].axis('off')
grid_fig.suptitle('GAN生成画像のサンプル（50枚）\n多様性はあるか？同じような画像ばかりではないか？',
                  fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_305_04_diversity_check.png', dpi=150, bbox_inches='tight')
plt.show()

# ペアワイズ距離のヒストグラム
fig2, ax2 = plt.subplots(1, 1, figsize=(10, 5))
ax2.hist(pairwise_dists, bins=40, color='steelblue', alpha=0.7, edgecolor='gray')
ax2.set_xlabel('画像間のMSE距離', fontsize=12)
ax2.set_ylabel('頻度', fontsize=12)
ax2.set_title('生成画像のペアワイズ距離分布\n右に広がるほど多様性が高い',
              fontsize=14, fontweight='bold')
ax2.axvline(np.mean(pairwise_dists), color='red', linestyle='--',
            label=f'平均: {np.mean(pairwise_dists):.4f}')
ax2.legend(fontsize=11)
plt.tight_layout()
plt.savefig('fig_305_05_diversity_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"💡 ペアワイズ距離:")
print(f"   平均: {np.mean(pairwise_dists):.4f}")
print(f"   標準偏差: {np.std(pairwise_dists):.4f}")
print(f"   モード崩壊が起きていれば、距離が全体的に小さくなる")

---

## まとめ

### 🎯 このノートブックで学んだこと

**GANの潜在空間の特性**
- ✓ GANはエンコーダがないため、画像→z変換には追加手法が必要
- ✓ 画像品質はVAEより高いが、潜在空間の構造保証がない

**Z空間 vs W空間**
- ✓ Z空間は等方的正規分布（入力の制約）
- ✓ W空間（StyleGAN）はMapping Networkでデータに適応した分布に変換
- ✓ W空間のほうがDisentanglementが高く、操作性に優れる

**モード崩壊**
- ✓ データの一部のモードしか生成しなくなる現象
- ✓ 潜在空間の「空白領域」を生み、補間品質を低下させる

### ⚠️ よくある間違い

❌ 「GANの潜在空間はVAEより劣っている」
✅ 構造の保証はないが、高品質な画像生成と操作性ではGAN（特にStyleGAN）が優れている場合が多い

❌ 「Z空間とW空間は同じもの」
✅ Z空間は固定の正規分布、W空間は学習された非線形変換後の空間で、構造が大きく異なる

### ✅ 学習チェックリスト

- [ ] GANとVAEの潜在空間の違いを3つ以上挙げられるか？
- [ ] W空間がZ空間より操作性に優れる理由を説明できるか？
- [ ] モード崩壊が潜在空間にどう影響するか説明できるか？

---

**次のステップ**: ノートブック306「潜在空間の線形補間とSlerp」で、高品質な補間手法を学びます！

---

## 🎓 自己評価クイズ

### Q1: GANにはエンコーダがないのに、なぜ潜在空間での画像操作が可能なのか？

<details>
<summary>💡 答えを見る</summary>

**答え**: Generatorが z→画像 の写像を学習しているため、zの操作（補間、加算等）で画像を制御できる

ただし、実画像を操作するにはGAN Inversion（画像→zの逆算）が別途必要です。これは309章で学びます。

</details>

---

### Q2: StyleGANのMapping Networkの役割は何か？

<details>
<summary>💡 答えを見る</summary>

**答え**: 固定の正規分布（Z空間）を、データの構造に適応した分布（W空間）に非線形変換する

8層のMLPにより、等方的な球状分布が、データの意味的な構造を反映した歪んだ分布に変換されます。これにより属性の分離が促進され、操作性が向上します。

</details>

---

### Q3: モード崩壊が起きたとき、潜在空間の補間はどうなるか？

<details>
<summary>💡 答えを見る</summary>

**答え**: 補間経路上で生成されないモードに対応する領域を通ると、品質が大幅に低下する

例えば数字1と7しか生成しないGANで1→7の補間をすると、途中で他の数字が本来現れるべきだが、代わりに崩れた画像や1/7の中間的な不自然な画像が生成される。

</details>